# 1. Setup OpenMeteo Data Source

In [32]:
%pip install openmeteo-requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
%pip install requests-cache retry-requests numpy pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 52.52,
	"longitude": 13.41,
	"hourly": "temperature_2m",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

hourly_data["temperature_2m"] = hourly_temperature_2m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Coordinates: 52.52000045776367°N 13.419998168945312°E
Elevation: 38.0 m asl
Timezone difference to GMT+0: 0s

Hourly data
                          date  temperature_2m
0   2026-07-17 00:00:00+00:00       22.945499
1   2026-07-17 01:00:00+00:00       22.495501
2   2026-07-17 02:00:00+00:00       21.445499
3   2026-07-17 03:00:00+00:00       20.495501
4   2026-07-17 04:00:00+00:00       20.545500
..                        ...             ...
163 2026-07-23 19:00:00+00:00       21.969501
164 2026-07-23 20:00:00+00:00       20.969501
165 2026-07-23 21:00:00+00:00       19.969501
166 2026-07-23 22:00:00+00:00       19.069500
167 2026-07-23 23:00:00+00:00       18.169500

[168 rows x 2 columns]


In [35]:
# Đọc danh sách đơn vị hành chính và tọa độ từ Parquet.
from pathlib import Path

# Hỗ trợ chạy notebook từ thư mục gốc hoặc trực tiếp từ thư mục data/.
parquet_candidates = [Path("data/dien_bien_locations.parquet"), Path("dien_bien_locations.parquet")]
locations_path = next((path for path in parquet_candidates if path.exists()), None)
if locations_path is None:
    raise FileNotFoundError("Không tìm thấy data/dien_bien_locations.parquet")

locations_df = pd.read_parquet(locations_path).rename(
    columns={"new_admin_unit": "admin_unit", "old_admin_unit": "admin_unit_old"}
)

locations = locations_df.to_dict(orient="records")
print(f"Đã đọc {len(locations)} địa điểm từ {locations_path}")

weather_params = {
    "latitude": [place["latitude"] for place in locations],
    "longitude": [place["longitude"] for place in locations],
    "hourly": [
        "temperature_2m", "relative_humidity_2m", "precipitation",
        "weather_code", "wind_speed_10m", "visibility",
    ],
    "timezone": "Asia/Ho_Chi_Minh",
    "forecast_days": 7,
}

responses = openmeteo.weather_api(url, params=weather_params)
frames = []
variables = weather_params["hourly"]

for place, response in zip(locations, responses):
    hourly = response.Hourly()
    frame = pd.DataFrame({
        "time": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left",
        ).tz_convert("Asia/Ho_Chi_Minh"),
        **{name: hourly.Variables(index).ValuesAsNumpy() for index, name in enumerate(variables)},
    })
    frame.insert(0, "admin_unit", place["admin_unit"])
    frame.insert(1, "admin_unit_old", place["admin_unit_old"])
    frame.insert(2, "province", "Điện Biên")
    frame["latitude"] = response.Latitude()
    frame["longitude"] = response.Longitude()
    frames.append(frame)

dien_bien_weather_df = (
    pd.concat(frames, ignore_index=True)
    .sort_values(["time", "admin_unit"], ignore_index=True)
)

# Xem cùng một thời điểm ở tất cả điểm để xác nhận dữ liệu không bị lặp một địa danh.\n
first_time = dien_bien_weather_df["time"].min()
display(dien_bien_weather_df.loc[dien_bien_weather_df["time"] == first_time])
print(f"{len(dien_bien_weather_df):,} dòng | {dien_bien_weather_df['admin_unit'].nunique()} xã/phường")


Đã đọc 85 địa điểm từ dien_bien_locations.parquet


,admin_unit,admin_unit_old,province,time,temperature_2m,relative_humidity_2m,precipitation,weather_code,wind_speed_10m,visibility,latitude,longitude
0,Phường Mường Thanh,Phường Noong Bua,Điện Biên,2026-07-17 00:00:00+07:00,24.267000,93.876022,2.2,80.0,1.049571,3400.0,21.335676,103.027519
1,Phường Mường Thanh,Phường Nam Thanh,Điện Biên,2026-07-17 00:00:00+07:00,24.358000,93.880035,2.2,80.0,1.049571,3400.0,21.335676,103.027519
2,Phường Mường Thanh,Xã Thanh Xương,Điện Biên,2026-07-17 00:00:00+07:00,24.442501,93.883797,2.2,80.0,1.049571,3400.0,21.335676,103.027519
3,Phường Điện Biên Phủ,Phường Mường Thanh,Điện Biên,2026-07-17 00:00:00+07:00,24.338501,93.879173,2.2,80.0,1.049571,3400.0,21.335676,103.027519
4,Phường Điện Biên Phủ,Phường Tân Thanh,Điện Biên,2026-07-17 00:00:00+07:00,24.151501,96.456146,2.2,80.0,2.012461,3380.0,21.405973,103.040817
...,...,...,...,...,...,...,...,...,...,...,...,...
80,Xã Tủa Chùa,Thị trấn Tủa Chùa,Điện Biên,2026-07-17 00:00:00+07:00,21.900000,97.883995,1.5,80.0,2.675892,2080.0,21.898066,103.412125
81,Xã Tủa Chùa,Xã Mường Báng,Điện Biên,2026-07-17 00:00:00+07:00,17.948999,99.372871,3.5,95.0,4.175069,2060.0,21.968365,103.333328
82,Xã Tủa Chùa,Xã Nà Tòng,Điện Biên,2026-07-17 00:00:00+07:00,19.177500,99.378700,3.5,95.0,4.175069,2060.0,21.968365,103.333328
83,Xã Xa Dung,Xã Xa Dung,Điện Biên,2026-07-17 00:00:00+07:00,23.909500,98.804108,1.1,55.0,2.346913,3920.0,21.335676,103.394493


14,280 dòng | 30 xã/phường


In [36]:
dien_bien_weather_df.dtypes

admin_unit                                        object
admin_unit_old                                    object
province                                          object
time                    datetime64[ns, Asia/Ho_Chi_Minh]
temperature_2m                                   float32
relative_humidity_2m                             float32
precipitation                                    float32
weather_code                                     float32
wind_speed_10m                                   float32
visibility                                       float32
latitude                                         float64
longitude                                        float64
dtype: object

In [37]:
display(dien_bien_weather_df.head(200))

,admin_unit,admin_unit_old,province,time,temperature_2m,relative_humidity_2m,precipitation,weather_code,wind_speed_10m,visibility,latitude,longitude
0,Phường Mường Thanh,Phường Noong Bua,Điện Biên,2026-07-17 00:00:00+07:00,24.267000,93.876022,2.2,80.0,1.049571,3400.0,21.335676,103.027519
1,Phường Mường Thanh,Phường Nam Thanh,Điện Biên,2026-07-17 00:00:00+07:00,24.358000,93.880035,2.2,80.0,1.049571,3400.0,21.335676,103.027519
2,Phường Mường Thanh,Xã Thanh Xương,Điện Biên,2026-07-17 00:00:00+07:00,24.442501,93.883797,2.2,80.0,1.049571,3400.0,21.335676,103.027519
3,Phường Điện Biên Phủ,Phường Mường Thanh,Điện Biên,2026-07-17 00:00:00+07:00,24.338501,93.879173,2.2,80.0,1.049571,3400.0,21.335676,103.027519
4,Phường Điện Biên Phủ,Phường Tân Thanh,Điện Biên,2026-07-17 00:00:00+07:00,24.151501,96.456146,2.2,80.0,2.012461,3380.0,21.405973,103.040817
...,...,...,...,...,...,...,...,...,...,...,...,...
195,Xã Mường Toong,Xã Mường Toong,Điện Biên,2026-07-17 02:00:00+07:00,22.290998,98.488953,0.6,53.0,2.052316,2200.0,22.038664,102.605560
196,Xã Mường Toong,Xã Huổi Lếch,Điện Biên,2026-07-17 02:00:00+07:00,21.718500,97.282982,0.4,51.0,2.747581,2680.0,22.108961,102.711342
197,Xã Mường Tùng,Xã Mường Tùng,Điện Biên,2026-07-17 02:00:00+07:00,20.203001,99.383514,1.1,55.0,0.569210,2820.0,21.898066,103.134636
198,Xã Mường Tùng,Xã Huổi Lèng,Điện Biên,2026-07-17 02:00:00+07:00,17.734499,97.816551,0.8,53.0,3.438895,3220.0,21.827766,103.121155


In [38]:
print("Bắt đầu:", dien_bien_weather_df["time"].min())
print("Kết thúc:", dien_bien_weather_df["time"].max())
print("Số mốc thời gian:", dien_bien_weather_df["time"].nunique())
print("Số địa điểm:", dien_bien_weather_df["admin_unit_old"].nunique())
print("Tổng số dòng:", len(dien_bien_weather_df))

Bắt đầu: 2026-07-17 00:00:00+07:00
Kết thúc: 2026-07-23 23:00:00+07:00
Số mốc thời gian: 168
Số địa điểm: 85
Tổng số dòng: 14280


In [39]:
# Vertical slice: elevation → forecast snapshot → cảnh báo MVP.
%pip install pyarrow requests

import subprocess
import sys
from pathlib import Path
import pandas as pd

data_dir = Path("data") if Path("data").exists() else Path(".")
locations_path = data_dir / "dien_bien_locations.parquet"
if not locations_path.exists():
    raise FileNotFoundError(f"Không tìm thấy {locations_path}")

for script_name in [
    "download_elevation.py",
    "download_forecast.py",
    "alert_rules.py",
    "verify_weather_alert_mvp.py",
]:
    script_path = data_dir / script_name
    subprocess.run([sys.executable, str(script_path)], check=True)

# P3: điểm sông OSM → GloFAS daily → tín hiệu xu hướng (không phải cảnh báo chính thức).
river_points_path = data_dir / "river_points.parquet"
if not river_points_path.exists():
    subprocess.run(
        [sys.executable, str(data_dir / "build_river_points.py")],
        check=True,
    )
for script_name in [
    "download_flood.py",
    "flood_signal.py",
    "verify_flood_signal.py",
]:
    subprocess.run(
        [sys.executable, str(data_dir / script_name)],
        check=True,
    )

overview_files = sorted(
    (data_dir / "alerts").glob(
        "snapshot_date=*/snapshot_time=*/new_admin_risk_overview.parquet"
    )
)
if not overview_files:
    raise FileNotFoundError("Chưa có new_admin_risk_overview.parquet")

risk_overview = pd.read_parquet(overview_files[-1])
flood_signals = pd.read_parquet(data_dir / "flood_signals.parquet")
display(
    risk_overview.sort_values(
        ["severity_rank", "new_admin_unit"],
        ascending=[False, True],
    )
)
display(flood_signals.sort_values("peak_change_percent", ascending=False))



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


CalledProcessError: Command '['c:\\Users\\tranq\\AppData\\Local\\Programs\\Python\\Python313\\python.exe', 'download_elevation.py']' returned non-zero exit status 1.